# Assignment DSC400 5.2

**Author:** Yisrael Wasserman  
**Date:** October 11, 2026  
**Description:** Apache Spark 

# Summary of Data Exploration and Analysis


# Data Processing and Analysis Summary

### 1. Data Exploration
I loaded the Customer Interactions Dataset into **Apache Spark** and examined its structure, schema, descriptive statistics, missing values, and unique categories.

The dataset contained 10,000 records with no missing values or obvious formatting issues.

### 2. Data Cleaning
I investigated several potential anomalies:

- Unusual event frequencies, including purchases being nearly as common as views.
- 1,190 Safari/browser-OS incompatibilities (11.9%).
- 53 repeated session IDs, some belonging to different customers.

I retained the browser mismatches because they appeared systematic, but removed the 106 records associated with repeated session IDs, creating `df_clean` with 9,894 records.

### 3. Feature Engineering
I created two features to support further analysis:

- **`Browser_OS_Mismatch`:** Identifies incompatible Safari/OS combinations.
- **`Date`:** Extracts the date from timestamps for time-based aggregations.

### 4. Spark SQL and Advanced Aggregations
I used **Spark SQL** to analyze event frequencies and browser/OS combinations.

I also used PySpark aggregations to calculate distinct sessions per customer, examine event frequencies by session count, and group interactions by date.

### 5. Visualization and Reporting
I created Matplotlib charts showing daily event frequency and daily purchases.

Overall activity remained relatively stable throughout 2021, with no obvious long-term trend.

### Conclusion
While the initial exploration revealed few issues, deeper analysis uncovered several inconsistencies suggesting that the dataset may be incomplete or synthetically generated without accounting for realistic customer behavior.

# 1. Loading the Dataset:

In [1]:
from pyspark.sql import SparkSession

In [29]:
spark = SparkSession.builder.getOrCreate()
# Load the dataset
df = spark.read.csv('customer_interactions.csv', inferSchema=True, header=True)

# 2. Explore the dataset: 

In [3]:
df.show(5)

+----------+---------+-------------------+---------+-------+-------+
|CustomerID|EventType|     EventTimestamp|SessionID|Browser|     OS|
+----------+---------+-------------------+---------+-------+-------+
|     12147| purchase|2021-07-16 16:23:30|   940350|  Opera|Windows|
|     16521|     view|2021-02-12 14:59:19|   707398|   Edge|Windows|
|     13215| purchase|2021-07-21 15:40:10|   495004|Firefox|    iOS|
|     11089|     view|2021-03-26 04:02:13|   963576|Firefox|Android|
|     18626|     view|2021-05-14 02:11:45|   796383|Firefox|Windows|
+----------+---------+-------------------+---------+-------+-------+
only showing top 5 rows


6 columns, that seem log customer interactions on a website. 

Logging time and some customer details such as ID, browser and OS. 

In [4]:
df.printSchema()

root
 |-- CustomerID: integer (nullable = true)
 |-- EventType: string (nullable = true)
 |-- EventTimestamp: timestamp (nullable = true)
 |-- SessionID: integer (nullable = true)
 |-- Browser: string (nullable = true)
 |-- OS: string (nullable = true)



All dtypes seem correct

In [5]:
df.describe().show(truncate=False)

26/10/11 12:36:38 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-------+-----------------+---------+------------------+-------+-------+
|summary|CustomerID       |EventType|SessionID         |Browser|OS     |
+-------+-----------------+---------+------------------+-------+-------+
|count  |10000            |10000    |10000             |10000  |10000  |
|mean   |15022.5684       |NULL     |546851.6581       |NULL   |NULL   |
|stddev |2894.248682588463|NULL     |259465.87716358845|NULL   |NULL   |
|min    |10000            |click    |100022            |Chrome |Android|
|max    |19999            |view     |999637            |Safari |macOS  |
+-------+-----------------+---------+------------------+-------+-------+



Looks like 10k enties, nothing else of note here. 

In [6]:
#check for null
from pyspark.sql.functions import col, count, when

df.select([
    count(when(col(c).isNull(), c)).alias(c) # alias renames the output columns
    for c in df.columns # loop through columns
]).show()

+----------+---------+--------------+---------+-------+---+
|CustomerID|EventType|EventTimestamp|SessionID|Browser| OS|
+----------+---------+--------------+---------+-------+---+
|         0|        0|             0|        0|      0|  0|
+----------+---------+--------------+---------+-------+---+



No null entries 

In [7]:
# view unique 
df.select("OS").distinct().show()
df.select("Browser").distinct().show()
df.select("EventType").distinct().show()

+-------+
|     OS|
+-------+
|    iOS|
|  Linux|
|  macOS|
|Android|
|Windows|
+-------+

+-------+
|Browser|
+-------+
|Firefox|
| Safari|
| Chrome|
|   Edge|
|  Opera|
+-------+

+---------+
|EventType|
+---------+
|   logout|
|    login|
| purchase|
|     view|
|    click|
+---------+



There are five event types. I would hypothesize that their frequencies follow this order:

    

**Clicks > Views > Logins ≥ Logouts > Purchases**

My reasoning is that users may click multiple times during a session, while views are generally less frequent. Logins and logouts should occur at roughly similar rates, although some users might leave without explicitly logging out. Finally, purchases should be the least frequent event, since only a fraction of user activity typically results in a purchase.

This could be a useful way to sanity-check the dataset later by comparing the actual event frequencies against my expectations.


Also I expect all Safari sessions to be on MacOs since its an Apple browser. (Similar rule for Edge with Windows) 

## Initial Data Exploration results: 

Everything looks good so far.

- Data types are correct.
- No null values were found.
- All values in `EventType`, `Browser`, and `OS` seem plausible.



# 3. Deeper exploration:



### Deeper Exploration

I've identified two ways to sanity-check the dataset:

1. **Event frequencies:** Compare the actual counts of each `EventType` against what I would expect based on typical user behavior.
2. **OS and browser combinations:** Check for unlikely or incompatible combinations of operating systems and browsers.

A few additional ideas for exploring user behavior:

1. **Most active sessions:** Identify the sessions with the most activity using `SessionID` and examine their usage patterns.
2. **Top purchasing customers:** Identify the customers who make the most purchases and look for common behavioral patterns.

## a. Event frequencies

In [8]:
# see count of each EventType
df.createOrReplaceTempView("C") 

spark.sql("""
    SELECT EventType, COUNT(*) AS Total
    FROM C
    GROUP BY EventType
    ORDER BY Total DESC
""").show()

+---------+-----+
|EventType|Total|
+---------+-----+
|     view| 2064|
|   logout| 2020|
|    click| 1995|
|    login| 1977|
| purchase| 1944|
+---------+-----+



### Some potential issues stand out:

1. **Logouts > Logins:** This seems unlikely, since users generally need to log in before they can log out.

2. **Views > Clicks:** I initially assumed that users would need to click something before viewing a page. However, that might not be true. Users could arrive through direct links, Google searches, or shared URLs. It's also possible that the dataset only records clicks made within the website.

3. **Purchases are 94% of views:** This seems like an unusually high purchase rate and raises questions about how these events are being recorded.

### Conclusion
These results don't necessarily indicate errors in the dataset. They could simply reflect differences between how events are defined or tracked and what I initially assumed.

## b. Browser to OS compatability 

In [9]:
spark.sql("""
    SELECT OS, Browser, COUNT(*) AS count
    FROM C
    GROUP BY OS, Browser
    ORDER BY OS ASC, count DESC
""").show(100, truncate=False)

+-------+-------+-----+
|OS     |Browser|count|
+-------+-------+-----+
|Android|Firefox|440  |
|Android|Opera  |427  |
|Android|Safari |427  |
|Android|Chrome |412  |
|Android|Edge   |396  |
|Linux  |Edge   |438  |
|Linux  |Safari |401  |
|Linux  |Chrome |393  |
|Linux  |Opera  |378  |
|Linux  |Firefox|356  |
|Windows|Firefox|426  |
|Windows|Opera  |410  |
|Windows|Chrome |403  |
|Windows|Edge   |385  |
|Windows|Safari |362  |
|iOS    |Edge   |434  |
|iOS    |Opera  |396  |
|iOS    |Safari |387  |
|iOS    |Firefox|382  |
|iOS    |Chrome |373  |
|macOS  |Chrome |409  |
|macOS  |Opera  |406  |
|macOS  |Firefox|403  |
|macOS  |Edge   |392  |
|macOS  |Safari |364  |
+-------+-------+-----+



All operating systems appear to use every browser, with negligible differences in frequency between them.

This is unrealistic, since Safari is not available on modern non-Apple operating systems.

In [10]:
# make a mismatch column to count the amount of times there was a mismatch for safari with anything other than Apple OS (ie. iSO, macOS):  

df = df.withColumn(
    "Browser_OS_Mismatch",
    when(
        (col("Browser") == "Safari") &
        (~col("OS").isin("macOS", "iOS")),
        "Incompatible"
    ).otherwise("Compatiable")
)

In [11]:
df.groupBy("Browser_OS_Mismatch").count().show()

+-------------------+-----+
|Browser_OS_Mismatch|count|
+-------------------+-----+
|        Compatiable| 8810|
|       Incompatible| 1190|
+-------------------+-----+



From here we can see that the amount of times there was a missmatch is notable, at 11.9% (1190/10000) of the database.

## Conclusion of Section Two

The dataset failed both of my sanity checks. 

The first discrepancy (event frequencies) could simply be due to differences in how events are logged. 

However, the second issue (OS and browser mismatches) is more concerning, as some of the combinations are not realistically possible.

This mismatch suggests that the dataset may be synthetic or that the OS and browser values were generated independently, without accounting for compatibility.

# 4. User interactions: 

## a. Frequency of return customers: 

In [12]:
from pyspark.sql.functions import col, countDistinct

customer_sessions = (
    df.groupBy("CustomerID")
      .agg(countDistinct("SessionID").alias("SessionCount"))
)

# How many customers have 1, 2, 3, etc. sessions?
customer_freqency_table = customer_sessions.groupBy("SessionCount").count().orderBy("SessionCount")


customer_freqency_table.show()

+------------+-----+
|SessionCount|count|
+------------+-----+
|           1| 3706|
|           2| 1794|
|           3|  639|
|           4|  150|
|           5|   33|
|           6|    4|
+------------+-----+



This data seems good, as expected more frequent returns is less common but still exsists. 
- Notably the data has a significant % increase drop from each group above it. 

In [13]:
# Lets view each event type count for each SessionCount:

df_grouped = df.join(customer_sessions, on="CustomerID")

event_counts = (
    df_grouped.groupBy("SessionCount")
      .pivot("EventType")
      .count()
      .orderBy("SessionCount")
)

event_counts.show()

+------------+-----+-----+------+--------+----+
|SessionCount|click|login|logout|purchase|view|
+------------+-----+-----+------+--------+----+
|           1|  724|  733|   761|     713| 775|
|           2|  714|  702|   728|     705| 739|
|           3|  403|  381|   368|     372| 393|
|           4|  116|  121|   125|     115| 123|
|           5|   29|   34|    34|      36|  32|
|           6|    9|    6|     4|       3|   2|
+------------+-----+-----+------+--------+----+



## b. Most active sessions

In [14]:
top_10_users = (customer_sessions.orderBy(col("SessionCount").desc()))

In [15]:
for row in top_10_users.limit(10).collect():
    customer_id = row["CustomerID"]

    print("Customer ID:", customer_id)

    df.filter(col("CustomerID") == customer_id).orderBy("EventTimestamp").show(truncate=False)

Customer ID: 16601
+----------+---------+-------------------+---------+-------+-------+-------------------+
|CustomerID|EventType|EventTimestamp     |SessionID|Browser|OS     |Browser_OS_Mismatch|
+----------+---------+-------------------+---------+-------+-------+-------------------+
|16601     |purchase |2021-01-16 08:08:07|608963   |Edge   |Linux  |Compatiable        |
|16601     |purchase |2021-04-14 21:29:37|934896   |Edge   |Android|Compatiable        |
|16601     |click    |2021-04-18 01:03:27|849267   |Opera  |iOS    |Compatiable        |
|16601     |click    |2021-06-10 19:07:30|415851   |Chrome |Android|Compatiable        |
|16601     |click    |2021-06-11 17:33:48|103826   |Opera  |iOS    |Compatiable        |
|16601     |login    |2021-12-04 19:50:58|300343   |Firefox|Linux  |Compatiable        |
+----------+---------+-------------------+---------+-------+-------+-------------------+

Customer ID: 11373
+----------+---------+-------------------+---------+-------+-------+---

1. I don't see one repeating date.
2. I don't see any repeating sessionIDs
3. All users rearly change browsers and OS systems.
   
The data seems unusually random. Customers frequently switch between operating systems and browsers, sometimes using a different combination every session. 

Overall, the data looks more randomly generated than representative of real customer behavior. However, missing events or incomplete session records could explain some of these inconsistencies.

In [16]:
# sanity check, see if Session IDs repeat:

print(f"Total cases of repeating SessionIDs: {df.groupBy("SessionID").count().filter("count > 1").count()}\n")
df.groupBy("SessionID").count().filter("count > 1").show()

Total cases of repeating SessionIDs: 53

+---------+-----+
|SessionID|count|
+---------+-----+
|   170104|    2|
|   645533|    2|
|   261018|    2|
|   239302|    2|
|   997089|    2|
|   195458|    2|
|   349115|    2|
|   249772|    2|
|   731703|    2|
|   701289|    2|
|   313567|    2|
|   515471|    2|
|   927876|    2|
|   468020|    2|
|   921213|    2|
|   219375|    2|
|   183370|    2|
|   736038|    2|
|   903424|    2|
|   709824|    2|
+---------+-----+
only showing top 20 rows


ok so we do have some cases of repeating session IDs

In [17]:
# Find repeated SessionIDs
repeated_sessions = (
    df.groupBy("SessionID")
      .count()
      .filter("count > 1")
      .orderBy(col("count").desc())
)


# Show full details for the top 3
for row in repeated_sessions.limit(3).collect():
    session_id = row["SessionID"]

    print("\nSessionID:", session_id)
    df.filter(col("SessionID") == session_id) \
      .orderBy("EventTimestamp") \
      .show(truncate=False)


SessionID: 170104
+----------+---------+-------------------+---------+-------+-----+-------------------+
|CustomerID|EventType|EventTimestamp     |SessionID|Browser|OS   |Browser_OS_Mismatch|
+----------+---------+-------------------+---------+-------+-----+-------------------+
|10221     |click    |2021-03-27 07:18:32|170104   |Opera  |macOS|Compatiable        |
|16313     |click    |2021-06-13 13:19:58|170104   |Opera  |iOS  |Compatiable        |
+----------+---------+-------------------+---------+-------+-----+-------------------+


SessionID: 645533
+----------+---------+-------------------+---------+-------+-------+-------------------+
|CustomerID|EventType|EventTimestamp     |SessionID|Browser|OS     |Browser_OS_Mismatch|
+----------+---------+-------------------+---------+-------+-------+-------------------+
|13781     |logout   |2021-03-18 16:34:04|645533   |Chrome |Windows|Compatiable        |
|13936     |purchase |2021-08-06 03:01:39|645533   |Chrome |iOS    |Compatiable    

The sessionID is the same for 53 cases but it looks like the CustomerID is different, which doesnt make much sense. 

Also since this is such a small fraction of the dataset I will drop these 106 rows. 

In [18]:
# Remove all events with repeating SessionIDs
df_clean = df.join(
    repeated_sessions.select("SessionID"),
    on="SessionID",
    how="left_anti"
)

print("Original rows:", df.count())
print("Cleaned rows:", df_clean.count())
print("Rows removed:", df.count() - df_clean.count())

Original rows: 10000
Cleaned rows: 9894
Rows removed: 106


# 4. Time based analysis

In [28]:

import plotly.express as px
from pyspark.sql.functions import to_date

# Count events per day
daily = df_clean.withColumn("Date", to_date("EventTimestamp")) \
    .groupBy("Date") \
    .count() \
    .orderBy("Date")

# Get data for plotting
rows = daily.collect()

dates = [row["Date"] for row in rows]
counts = [row["count"] for row in rows]

# line chart
fig = px.line(
    x=dates,
    y=counts,
    labels={"x": "Date", "y": "Number of Events"},
    title="Daily Event Frequency"
)

fig.show(renderer="iframe")

This shows a consistent event occurrence over time for all of 2021, between 10-45 and never trending too much.  



## Lets now see purchases over time;

In [27]:

# Count purchases per day
daily = df_clean.filter(df["EventType"] == "purchase") \
    .withColumn("Date", to_date("EventTimestamp")) \
    .groupBy("Date") \
    .count() \
    .orderBy("Date")

# Get data for plotting
rows = daily.collect()

dates = [row["Date"] for row in rows]
counts = [row["count"] for row in rows]

# line chart
fig = px.line(
    x=dates,
    y=counts,
    labels={"x": "Date", "y": "Number of purchases"},
    title="Daily Purchase Frequency"
)

fig.show(renderer="iframe")

Daily purchase activity appears relatively consistent throughout the year, generally ranging between 2 and 12 purchases per day, with no clear overall trend.

Further analysis could reveal weekly, monthly, or hourly patterns that aren't visible in this chart. For now, daily fluctuations appear largely random.